# អេបប់ ០៣ - គំរូរចនាកម្ម Agentic

ក្នុងអេបប់នេះ យើងស្វែងយល់ពីគំរូរចនាកម្មមូលដ្ឋានបីសម្រាប់បង្កើតភ្នាក់ងារថ្មីៗ ដែលមានប្រសិទ្ធភាព៖

1. **សេចក្តីណែនាំភ្នាក់ងារបញ្ជាក់ច្បាស់** — សរសេរជារូបរាងត្រឹមត្រូវ ដែលកំណត់តួនាទី និងណែនាំអាកប្បកិរិយារបស់ភ្នាក់ងារ
2. **លទ្ធផលរចនាសម្ព័ន្ធជាមួយម៉ូដែល Pydantic** — ធ្វើឱ្យប្រាកដថាភ្នាក់ងារផ្តល់នូវទិន្នន័យដែលអាចទាយបាន និងបានផ្ទៀងផ្ទាត់
3. **ភ្នាក់ងារទទួលខុសត្រូវតែមួយ** — រចនាភ្នាក់ងារដែលផ្តោតលើការធ្វើមួយរឿងបានល្អ

យើងនឹងអនុវត្តន៍គំរូនីមួយៗលើសហគ្រាស **អ្នកផ្តល់អនុសាសន៍កន្លែងទេសចរណ៍** ដោយកសាងប្រព័ន្ធឱ្យអាចផ្តល់អនុសាសន៍កន្លែងទេសចរណ៍បាន ស្វែងរកការត្រូវការចាស់ និងដោះស្រាយការតម្រូវការផ្សេងៗ។


## ការតម្លើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## លំនាំទី 1៖ សេចក្ដីណែនាំជាក់លាក់សម្រាប់ភ្នាក់ងារ

លំនាំដែលមានឥទ្ធិពលខ្លាំងបំផុតគឺជាលំនាំដែលសាមញ្ញបំផុតផងដែរ៖ ការសរសេរសេចក្ដីណែនាំច្បាស់លាស់លម្អិតសម្រាប់ភ្នាក់ងាររបស់អ្នក។

សេចក្ដីណែនាំល្អកំណត់៖
- **នរណា** ដែលជាភ្នាក់ងារ (បុគ្គលិក និងសម្លេង)
- **អ្វីដែល** វាគួរត្រូវធ្វើ (បេសកកម្មតាមជំហ៊ាន)
- **របៀបដែល** វាគួរត្រូវអនុវត្ត (ការជ្រុះចេញ និងស្តាយលីង)

ខាងក្រោម យើងបង្កើតភ្នាក់ងារជំនួយការធ្វើដំណើរដោយមានសេចក្ដីណែនាំច្បាស់លាស់ដែលរៀបចំនូវការឆ្លើយតបរាល់អ្វីដែលវាស PRODUCES។


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## រចនាសម្ព័ន្ធ 2: លទ្ធផលមានរចនាសម្ព័ន្ធជាមួយម៉ូដែល Pydantic

អត្ថបទទ្រង់ទ្រាយសេរីមានប្រយោជន៍សម្រាប់ការជជែកជាភាសា តែប្រព័ន្ធខាងក្រោមត្រូវការទិន្នន័យមានរចនាសម្ព័ន្ធ។
ដោយនឹងភ្ជាប់ **ម៉ូដែល Pydantic** ជាមួយ **មុខងារ​ឧបករណ៍** យើងអាច៖

- កំណត់រចនាសម្ព័ន្ធត្រឹមត្រូវសម្រាប់លទ្ធផលរបស់ភ្នាក់ងារ
- ត្រួតពិនិត្យចម្លើយដោយស្វ័យប្រវត្តិ
- សម្របសម្រួលលទ្ធផលភ្នាក់ងារចូលក្នុងលូជីកកម្មវិធីយ៉ាងទៀងទាត់

យើងក៏ណែនាំឧបករណ៍មួយដែលត្រឡប់ព័ត៌មានលម្អិតពីគោលដៅ ដូច្នេះភ្នាក់ងារអាចគាំទ្រការផ្តល់អនុសាសន៍ឲ្យមានមូលដ្ឋានលើទិន្នន័យពិត។


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## លំនាំទី 3: អ្នកតំណាងមានភារកិច្ចតែមួយ

ភារកិច្ចស្មុគស្មាញទទួលបានអត្ថប្រយោជន៍ពីការបំបែកការងារជាសមាសភាគច្រើនដែលមានការផ្តោតលើភារកិច្ចតែមួយ៖

- **អ្នកជំនាញគោលដៅ** ដែលដឹងអំពីកន្លែងនានា និងការចូលដំណាក់កាល
- **អ្នករៀបចំគម្រោងសម្ភារៈចរាចរណ៍** ដែលដោះស្រាយការហោះហើរ សណ្ឋាគារ និងផែនការតាមផ្លូវ

នេះស្របទៅនឹងគោលនយោបាយវិស្វកម្មកម្មវិធី *ការបំបែកការចូលរួម* — អ្នកតំណាងនីមួយៗងាយស្រួលក្នុងការប្រឡង ទទួលបានការថែទាំ និងកែលម្អដោយឡែក។


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## សេចក្ដីសង្ខេប

ក្នុងមេរៀននេះ យើងបានអនុវត្តន៍លំនាំរចនាសម្ព័ន្ធមួយចំនួនសម្រាប់ភ្នាក់ងារទៅលើសถานการณ์ផ្តល់អនុសាសន៍ដំណើរកម្សាន្ត៖

| លំនាំ | គំនិតសំខាន់ | អត្ថប្រយោជន៍ |
|---|---|---|
| **ការណែនាំច្បាស់លាស់** | កំណត់បុគ្គលិកលក្ខណៈភ្នាក់ងារ, ភារៈការនិងកំណត់ខ្ពស់នៅដើម | សមត្ថភាពអភិបាលមានភាពស្របតាមគោលនយោបាយមួយច្បាប់ |
| **លទ្ធផលមានរចនាសម្ព័ន្ធ** | ប្រើម៉ូដែល Pydantic ជាទ្រង់ទ្រាយចម្លើយ | លទ្ធផលដែលបានផ្ទៀងផ្ទាត់ និងអាចអានដោយម៉ាស៊ីន |
| **ភារកិច្ចតែមួយ** | ផ្តល់ភារកិច្ចមួយផ្តោតសំខាន់សម្រាប់បុគ្គលិកនីមួយៗ | ងាយស្រួលក្នុងការសាកល្បង រក្សាទុក និងបង្កើតរួម |

លំនាំទាំងនេះនឹងផ្សំឡើងដោយស្វ័យប្រវត្តិ — អ្នកអាចបញ្ចូលការណែនាំច្បាស់លាស់ជាមួយលទ្ធផលមានរចនាសម្ព័ន្ធនៅក្នុងភ្នាក់ងារភារកិច្ចតែមួយ ដើម្បីសាងសង់ប្រព័ន្ធមានភាពរឹងមាំ និងរួចរាល់សម្រាប់ផលិតកម្ម។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
